# WordEmbedding 기반 텍스트 분석 연습
> [02_text_analysis.ipynb]의 develop

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_preprocessing.ipynb]
2. Word Embedding model을 이용하여 벡터화한다.
3. 입력된 문자열의 긍/부정을 판단한다. (유사도 활용)

In [2]:
import pandas as pd
from konlpy.tag import Okt
import re
# 1. 데이터셋 주소 설정 (인터넷에 있는 파일을 직접 가리킵니다)
url = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"

# 2. 데이터 불러오기 
# sep='\t': 데이터가 탭(Tab)으로 구분되어 있다는 뜻입니다.
# names: 데이터의 열(Column) 이름을 우리가 직접 정해줍니다.
df = pd.read_csv(url, sep='\t', names=['rating', 'content'])

# 3. 잘 들어왔는지 확인
print(df.head())

   rating                                            content
0       5                                            배공빠르고 굿
1       2                      택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2       5  아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3       2  선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4       5                  민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


In [4]:
stopwords = set(['의', '가', '이', '은', '들', '는'])

df.dropna(inplace=True)

# 한글이 아닌 데이터를 제거
df['content'] = df['content'].apply(lambda x: re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', x))

# 불용어 제거
df['content'] = df['content'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords]))

# 토큰화 (어간 추출)
okt = Okt()
df['tokens'] = df['content'].apply(lambda x: okt.morphs(x))

In [6]:
# word emedding model 사용해 벡터화
from gensim.models import Word2Vec
# 1. 문장 단위로 토큰화
sentences = [row.split() for row in df['content']]
# 2. Word2Vec 모델 학습
model = Word2Vec(
    sentences, 
    vector_size=100, 
    window=5,
    min_count=1,
    workers=4)

In [ ]:
positive_words = ['좋아요', '만족', '추천', '최고', '행복', '예쁨', '감사', '친절', '빠름', '편리',
                  '예뻐요', '맛있어요', '좋습니다', '최고에요', '행복해요', '감사합니다', '친절해요', '빠릅니다', '편리해요']

def predict_sentiment(text):
    tokens = text.split()
    positive_score = sum(1 for word in tokens if word in positive_words)
    return '긍정' if positive_score > 0 else '부정'

# 예측 결과 확인


긍정


In [14]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from konlpy.tag import Okt

okt = Okt()

positive_seeds = ['좋다', '만족', '추천', '최고', '행복', '예쁘다', '감사', '친절', '빠르다', '편리']
negative_seeds = ['별로', '불만', '최악', '실망', '느리다', '불편', '짜증', '환불', '나쁘다', '문제']

positive_seeds = [w for w in positive_seeds if w in model.wv]
negative_seeds = [w for w in negative_seeds if w in model.wv]

def sentence_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return None
    return np.mean(vectors, axis=0)

pos_vector = sentence_vector(positive_seeds, model)
neg_vector = sentence_vector(negative_seeds, model)

def predict_sentiment(text):
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', text)
    tokens = okt.morphs(text, stem=True)
    vec = sentence_vector(tokens, model)
    
    if vec is None:
        return "판단불가", 0.0, 0.0
    
    pos_sim = cosine_similarity([vec], [pos_vector])[0][0]
    neg_sim = cosine_similarity([vec], [neg_vector])[0][0]

    if pos_sim > neg_sim:
        return "긍정", pos_sim, neg_sim
    elif neg_sim > pos_sim:
        return "부정", pos_sim, neg_sim
    else:
        return "중립", pos_sim, neg_sim

# 테스트
text = "배송이 빠르고 친절해요"
result, pos_sim, neg_sim = predict_sentiment(text)

print(f"입력 텍스트: {text}")
print(f"토큰: {okt.morphs(text, stem=True)}")
print(f"예측 결과: {result}")
print(f"긍정 유사도: {pos_sim:.4f}")
print(f"부정 유사도: {neg_sim:.4f}")

입력 텍스트: 배송이 빠르고 친절해요
토큰: ['배송', '이', '빠르다', '친절하다']
예측 결과: 긍정
긍정 유사도: 0.7380
부정 유사도: 0.6572


In [15]:
# 데이터 사용 긍정 부정 컬럼 추가
df['pos_pred'] = df['content'].apply(lambda x: predict_sentiment(x)[0])
df['naive_pred'] = df['content'].apply(lambda x: predict_sentiment(x)[1])
df

,rating,content,tokens,pos_pred,naive_pred
0,5,배공빠르고 굿,"[배공, 빠르고, 굿]",긍정,0.976377
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,"[택배, 가, 엉망, 이네, 용, 저희, 집, 밑, 에, 층, 에, 말, 도, 없이...",부정,0.867039
2,5,아주좋아요 바지 정말 좋아서개 더 구매했어요 이가격에 대박입니다 바느질이 조금 엉성...,"[아주, 좋아요, 바지, 정말, 좋아서, 개, 더, 구매, 했어요, 이, 가격, 에...",부정,0.906948
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다 전화...,"[선물, 용, 으로, 빨리, 받아서, 전달, 했어야, 하는, 상품, 이었는데, 머그...",부정,0.858238
4,5,민트색상 예뻐요 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,"[민트, 색상, 예뻐요, 옆, 손잡이, 는, 거, 는, 용, 도로, 도, 사용, 되...",부정,0.895127
...,...,...,...,...,...
199995,2,장마라그런가 달지않아요,"[장마, 라, 그, 런가, 달, 지, 않아요]",부정,0.814304
199996,5,다이슨 케이스 구매했어요 다이슨 슈퍼소닉 드라이기 케이스 구매했어요가격 괜찮고 배송...,"[다이슨, 케이스, 구매, 했어요, 다이슨, 슈퍼소닉, 드라이기, 케이스, 구매, ...",부정,0.899914
199997,5,로드샾에서 사는것보다 세배 저렴하네요 ㅜㅜ 자주이용할께요,"[로드샾, 에서, 사는것보다, 세배, 저렴하네요, ㅜㅜ, 자주, 이, 용할께요]",부정,0.862604
199998,5,넘이쁘고 쎄련되보이네요,"[넘, 이쁘고, 쎄련, 되, 보이네요]",부정,0.850372


In [16]:
df.to_csv('./data/sentiment_predictions.csv', index=False)